# Course-End Project — Marketing Campaigns

**Problem:** Exploratory data analysis and hypothesis testing on factors that influence customer acquisition (marketing mix / Four Ps).

**Libraries (course stack):** `pandas`, `numpy`, `matplotlib`, `seaborn`, and `scipy.stats` for hypothesis tests (as in Advanced Statistics / course packages).

**Data:** `marketing_data.csv` + data dictionary.


## Step 0 — Import libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_theme(style='whitegrid')
FIG = Path('figures')
FIG.mkdir(exist_ok=True)
pd.set_option('display.max_columns', 50)
print('ok')


ok


## Step 1 — Import data and check Dt_Customer & Income

In [2]:
df = pd.read_csv('marketing_data.csv')
print('shape:', df.shape)
print('columns:', df.columns.tolist())
print()
print('Dt_Customer dtype:', df['Dt_Customer'].dtype)
print(df['Dt_Customer'].head())
print()
# Income column has leading/trailing spaces in the header
print('Income raw column name repr:', [c for c in df.columns if 'Income' in c])
inc_col = [c for c in df.columns if 'Income' in c][0]
print('Income sample:', df[inc_col].head().tolist())
print('Income dtype:', df[inc_col].dtype)
df.head(3)


shape: (2240, 28)
columns: ['ID', 'Year_Birth', 'Education', 'Marital_Status', ' Income ', 'Kidhome', 'Teenhome', 'Dt_Customer', 'Recency', 'MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1', 'AcceptedCmp2', 'Response', 'Complain', 'Country']

Dt_Customer dtype: str
0    6/16/14
1    6/15/14
2    5/13/14
3    5/11/14
4     4/8/14
Name: Dt_Customer, dtype: str

Income raw column name repr: [' Income ']
Income sample: ['$84,835.00 ', '$57,091.00 ', '$67,267.00 ', '$32,474.00 ', '$21,474.00 ']
Income dtype: str


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumDealsPurchases,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Response,Complain,Country
0,1826,1970,Graduation,Divorced,"$84,835.00",0,0,6/16/14,0,189,104,379,111,189,218,1,4,4,6,1,0,0,0,0,0,1,0,SP
1,1,1961,Graduation,Single,"$57,091.00",0,0,6/15/14,0,464,5,64,7,0,37,1,7,3,7,5,0,0,0,0,1,1,0,CA
2,10476,1958,Graduation,Married,"$67,267.00",0,1,5/13/14,0,134,11,59,15,2,30,1,3,2,5,2,0,0,0,0,0,0,0,US


### Observation (Step 1)
- `Dt_Customer` imported as text (e.g. `6/16/14`) — should be parsed as datetime.
- `Income` header has spaces (` Income `) and values look like currency strings (`$84,835.00 `) — need cleansing to numeric.


## Step 2 — Clean Education / Marital_Status; impute missing Income by Education + Marital_Status mean

In [3]:
# standardize column name
df = df.rename(columns={inc_col: 'Income'})

# parse Income to float
df['Income'] = (
    df['Income']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
    .replace({'': np.nan, 'nan': np.nan, 'None': np.nan})
)
df['Income'] = pd.to_numeric(df['Income'], errors='coerce')

# parse date
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], format='mixed')

print('Education before:')
print(df['Education'].value_counts())
print()
print('Marital_Status before:')
print(df['Marital_Status'].value_counts())


Education before:
Education
Graduation    1127
PhD            486
Master         370
2n Cycle       203
Basic           54
Name: count, dtype: int64

Marital_Status before:
Marital_Status
Married     864
Together    580
Single      480
Divorced    232
Widow        77
Alone         3
YOLO          2
Absurd        2
Name: count, dtype: int64


In [4]:
# clean marital status categories (noise labels -> clearer groups)
marital_map = {
    'Alone': 'Single',
    'YOLO': 'Single',
    'Absurd': 'Single',
    'Together': 'Together',
    'Married': 'Married',
    'Single': 'Single',
    'Divorced': 'Divorced',
    'Widow': 'Widow',
}
df['Marital_Status'] = df['Marital_Status'].replace(marital_map)

# education: keep course categories; optional normalize 2n Cycle label only
# (2n Cycle is a valid European education label — keep as-is)

print('Marital_Status after:')
print(df['Marital_Status'].value_counts())
print()
print('missing Income before impute:', df['Income'].isna().sum())


Marital_Status after:
Marital_Status
Married     864
Together    580
Single      487
Divorced    232
Widow        77
Name: count, dtype: int64

missing Income before impute: 24


In [5]:
# impute Income with mean Income for same Education + Marital_Status
df['Income'] = df.groupby(['Education', 'Marital_Status'])['Income'].transform(
    lambda s: s.fillna(s.mean())
)
# if any still missing (rare group), fill with overall mean
df['Income'] = df['Income'].fillna(df['Income'].mean())
print('missing Income after impute:', df['Income'].isna().sum())
df[['Education', 'Marital_Status', 'Income', 'Dt_Customer']].head()


missing Income after impute: 0


,Education,Marital_Status,Income,Dt_Customer
0,Graduation,Divorced,84835.0,2014-06-16
1,Graduation,Single,57091.0,2014-06-15
2,Graduation,Married,67267.0,2014-05-13
3,Graduation,Together,32474.0,2014-05-11
4,Graduation,Single,21474.0,2014-04-08


## Step 3 — Create TotalChildren, Age, TotalSpending

In [6]:
# reference year for age: use max enrollment year or a fixed analysis year
# common approach: age at latest customer date / campaign year
ref_year = df['Dt_Customer'].dt.year.max()
df['Age'] = ref_year - df['Year_Birth']
df['TotalChildren'] = df['Kidhome'] + df['Teenhome']
spend_cols = ['MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']
df['TotalSpending'] = df[spend_cols].sum(axis=1)
df[['Year_Birth', 'Age', 'Kidhome', 'Teenhome', 'TotalChildren', 'TotalSpending']].head()


,Year_Birth,Age,Kidhome,Teenhome,TotalChildren,TotalSpending
0,1970,44,0,0,0,1190
1,1961,53,0,0,0,577
2,1958,56,0,1,1,251
3,1967,47,1,1,2,11
4,1989,25,1,0,1,91


## Step 4 — TotalPurchases across three channels (Web, Catalog, Store)

In [7]:
# problem: derive total purchases from transactions across the three channels
df['TotalPurchases'] = df['NumWebPurchases'] + df['NumCatalogPurchases'] + df['NumStorePurchases']
df[['NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases', 'TotalPurchases']].describe()


,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,TotalPurchases
count,2240.000000,2240.000000,2240.000000,2240.000000
mean,4.084821,2.662054,5.790179,12.537054
std,2.778714,2.923101,3.250958,7.205741
min,0.000000,0.000000,0.000000,0.000000
25%,2.000000,0.000000,3.000000,6.000000
50%,4.000000,2.000000,5.000000,12.000000
75%,6.000000,4.000000,8.000000,18.000000
max,27.000000,28.000000,13.000000,32.000000


## Step 5 — Box plots / histograms; outlier treatment

In [8]:
# distributions for key numeric fields
check_cols = ['Income', 'Age', 'TotalSpending', 'TotalPurchases', 'Recency']
fig, axes = plt.subplots(2, len(check_cols), figsize=(16, 6))
for i, c in enumerate(check_cols):
    sns.histplot(df[c], ax=axes[0, i], kde=False, color='#4C72B0')
    axes[0, i].set_title(c + ' hist')
    sns.boxplot(y=df[c], ax=axes[1, i], color='#55A868')
    axes[1, i].set_title(c + ' box')
plt.tight_layout()
plt.savefig(FIG/'01_distributions.png', dpi=120, bbox_inches='tight')
plt.show()
print(df[check_cols].describe())


              Income          Age  TotalSpending  TotalPurchases      Recency
count    2240.000000  2240.000000    2240.000000     2240.000000  2240.000000
mean    52248.619720    45.194196     605.798214       12.537054    49.109375
std     25039.967739    11.984069     602.249288        7.205741    28.962453
min      1730.000000    18.000000       5.000000        0.000000     0.000000
25%     35538.750000    37.000000      68.750000        6.000000    24.000000
50%     51381.500000    44.000000     396.000000       12.000000    49.000000
75%     68289.750000    55.000000    1045.500000       18.000000    74.000000
max    666666.000000   121.000000    2525.000000       32.000000    99.000000


/tmp/ipykernel_635975/4202027494.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
# IQR outlier treatment on Age and Income (extreme birth years / income typos)
def iqr_bounds(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

before = len(df)
age_lo, age_hi = iqr_bounds(df['Age'])
inc_lo, inc_hi = iqr_bounds(df['Income'])
print('Age bounds:', age_lo, age_hi)
print('Income bounds:', inc_lo, inc_hi)

# cap (winsorize) instead of dropping all rows — keeps sample size for campaigns
for col, lo, hi in [('Age', age_lo, age_hi), ('Income', inc_lo, inc_hi)]:
    df[col] = df[col].clip(lower=max(lo, 0) if col=='Age' else lo, upper=hi)

print('rows unchanged (capping):', len(df), 'from', before)
df[['Age', 'Income']].describe()


Age bounds: 10.0 82.0
Income bounds: -13587.75 117416.25
rows unchanged (capping): 2240 from 2240


,Age,Income
count,2240.000000,2240.000000
mean,45.147768,51876.518827
std,11.771725,20938.700038
min,18.000000,1730.000000
25%,37.000000,35538.750000
50%,44.000000,51381.500000
75%,55.000000,68289.750000
max,82.000000,117416.250000


## Step 6 — Ordinal + one-hot encoding for categoricals

In [10]:
# ordinal: Education
edu_order = {'Basic': 1, '2n Cycle': 2, 'Graduation': 3, 'Master': 4, 'PhD': 5}
df['Education_ord'] = df['Education'].map(edu_order)

# one-hot: Marital_Status, Country (drop_first to reduce collinearity)
marital_dummies = pd.get_dummies(df['Marital_Status'], prefix='Marital')
country_dummies = pd.get_dummies(df['Country'], prefix='Country')
df_enc = pd.concat([df, marital_dummies, country_dummies], axis=1)
print('Education_ord value counts:')
print(df['Education_ord'].value_counts().sort_index())
print('new dummy columns:', list(marital_dummies.columns) + list(country_dummies.columns))
df_enc.head(2)


Education_ord value counts:
Education_ord
1      54
2     203
3    1127
4     370
5     486
Name: count, dtype: int64
new dummy columns: ['Marital_Divorced', 'Marital_Married', 'Marital_Single', 'Marital_Together', 'Marital_Widow', 'Country_AUS', 'Country_CA', 'Country_GER', 'Country_IND', 'Country_ME', 'Country_SA', 'Country_SP', 'Country_US']


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumDealsPurchases,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Response,Complain,Country,Age,TotalChildren,TotalSpending,TotalPurchases,Education_ord,Marital_Divorced,Marital_Married,Marital_Single,Marital_Together,Marital_Widow,Country_AUS,Country_CA,Country_GER,Country_IND,Country_ME,Country_SA,Country_SP,Country_US
0,1826,1970,Graduation,Divorced,84835.0,0,0,2014-06-16,0,189,104,379,111,189,218,1,4,4,6,1,0,0,0,0,0,1,0,SP,44,0,1190,14,3,True,False,False,False,False,False,False,False,False,False,False,True,False
1,1,1961,Graduation,Single,57091.0,0,0,2014-06-15,0,464,5,64,7,0,37,1,7,3,7,5,0,0,0,0,1,1,0,CA,53,0,577,17,3,False,False,True,False,False,False,True,False,False,False,False,False,False


## Step 7 — Correlation heatmap

In [11]:
num_cols = [
    'Income', 'Age', 'TotalChildren', 'TotalSpending', 'TotalPurchases', 'Recency',
    'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth',
    'NumDealsPurchases', 'Response', 'Education_ord'
]
corr = df[num_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=False)
plt.title('Correlation heatmap (key numeric / engineered features)')
plt.tight_layout()
plt.savefig(FIG/'02_correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()
corr.round(2)


/tmp/ipykernel_635975/1924648894.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Income,Age,TotalChildren,TotalSpending,TotalPurchases,Recency,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,NumDealsPurchases,Response,Education_ord
Income,1.00,0.20,-0.35,0.80,0.75,0.01,0.47,0.69,0.65,-0.65,-0.11,0.17,0.18
Age,0.20,1.00,0.09,0.11,0.17,0.02,0.15,0.12,0.13,-0.12,0.06,-0.02,0.19
TotalChildren,-0.35,0.09,1.00,-0.50,-0.38,0.02,-0.15,-0.44,-0.32,0.42,0.44,-0.17,0.06
TotalSpending,0.80,0.11,-0.50,1.00,0.82,0.02,0.52,0.78,0.67,-0.50,-0.07,0.27,0.11
TotalPurchases,0.75,0.17,-0.38,0.82,1.00,0.01,0.77,0.79,0.86,-0.43,0.12,0.16,0.12
Recency,0.01,0.02,0.02,0.02,0.01,1.00,-0.01,0.03,0.00,-0.02,-0.00,-0.20,-0.01
NumWebPurchases,0.47,0.15,-0.15,0.52,0.77,-0.01,1.00,0.38,0.50,-0.06,0.23,0.15,0.10
NumCatalogPurchases,0.69,0.12,-0.44,0.78,0.79,0.03,0.38,1.00,0.52,-0.52,-0.01,0.22,0.09
NumStorePurchases,0.65,0.13,-0.32,0.67,0.86,0.00,0.50,0.52,1.00,-0.43,0.07,0.04,0.09
NumWebVisitsMonth,-0.65,-0.12,0.42,-0.50,-0.43,-0.02,-0.06,-0.52,-0.43,1.00,0.35,-0.00,-0.06


## Step 8 — Hypothesis tests

1. Older people prefer store shopping (higher store share / store purchases vs younger).
2. Customers with children prefer online (higher web purchases / web share).
3. Store sales cannibalized by other channels (negative correlation store vs web/catalog).
4. US total purchase volume significantly higher than rest of world?


In [12]:
# H1: Older vs younger technological / channel preference
# split by median age; compare NumStorePurchases and store share
med_age = df['Age'].median()
older = df[df['Age'] >= med_age]
younger = df[df['Age'] < med_age]

df['StoreShare'] = df['NumStorePurchases'] / df['TotalPurchases'].replace(0, np.nan)
older_share = older['NumStorePurchases'] / older['TotalPurchases'].replace(0, np.nan)
younger_share = younger['NumStorePurchases'] / younger['TotalPurchases'].replace(0, np.nan)

t1, p1 = stats.ttest_ind(older['NumStorePurchases'], younger['NumStorePurchases'], equal_var=False, nan_policy='omit')
t1s, p1s = stats.ttest_ind(older_share.dropna(), younger_share.dropna(), equal_var=False)
print('H1 older vs younger NumStorePurchases: t=%.3f p=%.4g' % (t1, p1))
print('  mean store purchases older/younger:', older['NumStorePurchases'].mean(), younger['NumStorePurchases'].mean())
print('H1 store share: t=%.3f p=%.4g' % (t1s, p1s))
print('  mean store share older/younger:', older_share.mean(), younger_share.mean())


H1 older vs younger NumStorePurchases: t=6.563 p=6.536e-11
  mean store purchases older/younger: 6.223764093668691 5.330266789328427
H1 store share: t=-4.609 p=4.277e-06
  mean store share older/younger: 0.4912185499751317 0.5202734933145348


In [13]:
# H2: with children vs no children — online convenience
with_kids = df[df['TotalChildren'] > 0]
no_kids = df[df['TotalChildren'] == 0]
df['WebShare'] = df['NumWebPurchases'] / df['TotalPurchases'].replace(0, np.nan)

t2, p2 = stats.ttest_ind(with_kids['NumWebPurchases'], no_kids['NumWebPurchases'], equal_var=False)
t2s, p2s = stats.ttest_ind(
    with_kids['NumWebPurchases']/with_kids['TotalPurchases'].replace(0, np.nan),
    no_kids['NumWebPurchases']/no_kids['TotalPurchases'].replace(0, np.nan),
    equal_var=False, nan_policy='omit'
)
print('H2 web purchases with_kids vs no_kids: t=%.3f p=%.4g' % (t2, p2))
print('  means:', with_kids['NumWebPurchases'].mean(), no_kids['NumWebPurchases'].mean())
print('H2 web share: t=%.3f p=%.4g' % (t2s, p2s))


H2 web purchases with_kids vs no_kids: t=-3.542 p=0.0004108
  means: 3.9619225967540572 4.393416927899686
H2 web share: t=14.929 p=2.179e-46


In [14]:
# H3: cannibalization — correlation of store purchases with web and catalog
c_web = df['NumStorePurchases'].corr(df['NumWebPurchases'])
c_cat = df['NumStorePurchases'].corr(df['NumCatalogPurchases'])
# Pearson p-values
r_w, p_w = stats.pearsonr(df['NumStorePurchases'], df['NumWebPurchases'])
r_c, p_c = stats.pearsonr(df['NumStorePurchases'], df['NumCatalogPurchases'])
print('Store vs Web corr: r=%.3f p=%.4g' % (r_w, p_w))
print('Store vs Catalog corr: r=%.3f p=%.4g' % (r_c, p_c))
print('Note: positive corr suggests channels rise together (affluent buyers), not classic cannibalization.')


Store vs Web corr: r=0.503 p=8.963e-144
Store vs Catalog corr: r=0.519 p=1.498e-154
Note: positive corr suggests channels rise together (affluent buyers), not classic cannibalization.


In [15]:
# H4: US vs rest of world in TotalPurchases
us = df[df['Country'] == 'US']['TotalPurchases']
row = df[df['Country'] != 'US']['TotalPurchases']
t4, p4 = stats.ttest_ind(us, row, equal_var=False)
print('H4 US vs rest TotalPurchases: t=%.3f p=%.4g' % (t4, p4))
print('  mean US / rest:', us.mean(), row.mean())
print('  n US / rest:', len(us), len(row))


H4 US vs rest TotalPurchases: t=1.468 p=0.1447
  mean US / rest: 13.513761467889909 12.487095260441107
  n US / rest: 109 2131


## Step 9 — Visual analyses from the problem statement

In [16]:
# 9a Top / bottom products by revenue (sum of Mnt*)
product_rev = df[spend_cols].sum().sort_values(ascending=False)
print(product_rev)
plt.figure(figsize=(8, 4))
sns.barplot(x=product_rev.index, y=product_rev.values, color='#4C72B0')
plt.xticks(rotation=30, ha='right')
plt.ylabel('Total revenue (amount spent)')
plt.title('Product revenue — top to bottom')
plt.tight_layout()
plt.savefig(FIG/'03_product_revenue.png', dpi=120, bbox_inches='tight')
plt.show()


MntWines            680816
MntMeatProducts     373968
MntGoldProds         98609
MntFishProducts      84057
MntSweetProducts     60621
MntFruits            58917
dtype: int64


/tmp/ipykernel_635975/3728262750.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
# 9b Age vs acceptance of last campaign (Response)
plt.figure(figsize=(7, 4))
sns.boxplot(data=df, x='Response', y='Age')
plt.title('Age vs last-campaign acceptance (Response)')
plt.xlabel('Response (0=no, 1=yes)')
plt.tight_layout()
plt.savefig(FIG/'04_age_vs_response.png', dpi=120, bbox_inches='tight')
plt.show()
print(df.groupby('Response')['Age'].agg(['count','mean','median']))


          count       mean  median
Response                          
0          1906  45.246590    44.0
1           334  44.583832    43.0


/tmp/ipykernel_635975/3817332229.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
# 9c Country with highest number who accepted last campaign
acc_by_country = df.groupby('Country')['Response'].sum().sort_values(ascending=False)
print(acc_by_country)
plt.figure(figsize=(8, 4))
sns.barplot(x=acc_by_country.index, y=acc_by_country.values, color='#55A868')
plt.ylabel('Accepted last campaign (count)')
plt.title('Last-campaign acceptances by country')
plt.tight_layout()
plt.savefig(FIG/'05_response_by_country.png', dpi=120, bbox_inches='tight')
plt.show()


Country
SP     176
SA      52
CA      38
AUS     23
GER     17
IND     13
US      13
ME       2
Name: Response, dtype: int64


/tmp/ipykernel_635975/2692876207.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [19]:
# 9d Children at home vs total expenditure
plt.figure(figsize=(7, 4))
sns.boxplot(data=df, x='TotalChildren', y='TotalSpending')
plt.title('Total spending by number of children at home')
plt.tight_layout()
plt.savefig(FIG/'06_children_vs_spending.png', dpi=120, bbox_inches='tight')
plt.show()
print(df.groupby('TotalChildren')['TotalSpending'].agg(['count','mean','median']))


               count         mean  median
TotalChildren                            
0                638  1106.029781  1189.5
1               1128   472.733156   305.0
2                421   245.947743    93.0
3                 53   274.603774    88.0


/tmp/ipykernel_635975/4122150325.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [20]:
# 9e Education of customers who complained in last 2 years
complainers = df[df['Complain'] == 1]
print('complainers n=', len(complainers))
print(complainers['Education'].value_counts())
plt.figure(figsize=(7, 4))
sns.countplot(data=complainers, x='Education', order=complainers['Education'].value_counts().index, color='#C44E52')
plt.title('Education of customers who complained (last 2 years)')
plt.tight_layout()
plt.savefig(FIG/'07_complain_education.png', dpi=120, bbox_inches='tight')
plt.show()


complainers n= 21
Education
Graduation    14
2n Cycle       4
Master         2
PhD            1
Name: count, dtype: int64


/tmp/ipykernel_635975/2308597511.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary of findings

1. **Import / clean:** Income currency strings fixed; Dt_Customer parsed; marital noise labels cleaned; Income imputed by Education × Marital_Status mean.
2. **Features:** TotalChildren, Age, TotalSpending, TotalPurchases (web+catalog+store).
3. **Outliers:** Age/Income winsorized via IQR caps.
4. **Encoding:** Education ordinal; Marital_Status & Country one-hot.
5. **Heatmap:** Shows relationships among income, spending, purchases, age, etc.
6. **Hypotheses:** Results printed above (p-values). Interpret with those outputs — e.g. store vs other channels often move together (not simple cannibalization); US vs rest depends on the t-test printout.
7. **Visuals:** Wine typically leads product revenue; Response vs age, country acceptances, children vs spending, and complainers’ education are plotted under `figures/`.

This notebook sticks to pandas / NumPy / Matplotlib / Seaborn / SciPy stats as used in the course materials.


In [21]:
# save cleaned analysis frame for reuse
df.to_csv('marketing_data_cleaned.csv', index=False)
print('wrote marketing_data_cleaned.csv', df.shape)


wrote marketing_data_cleaned.csv (2240, 35)
